In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid')

TRADING_DAYS = 252
VAR_LEVEL = 0.05
ROOT = Path.cwd()

print('Setup complete')

Setup complete


# Advanced Analytics + Risk Metrics

This notebook performs advanced mutual fund analytics, including VaR/CVaR, rolling Sharpe, investor cohort analysis, SIP continuity, sector concentration, and a simple recommender.

In [2]:
nav = pd.read_csv(ROOT / 'data' / 'processed' / '02_nav_history_cleaned.csv')
transactions = pd.read_csv(ROOT / 'data' / 'processed' / '08_investor_transactions_cleaned.csv')
holdings = pd.read_csv(ROOT / 'data' / 'raw' / '09_portfolio_holdings.csv')
fund_master = pd.read_csv(ROOT / 'data' / 'raw' / '01_fund_master.csv')
scheme_performance = pd.read_csv(ROOT / 'data' / 'processed' / '07_scheme_performance_cleaned.csv')

for name, frame in [('nav', nav), ('transactions', transactions), ('holdings', holdings), ('fund_master', fund_master), ('scheme_performance', scheme_performance)]:
    print(name, frame.shape)

nav['date'] = pd.to_datetime(nav['date'])
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])
holdings['portfolio_date'] = pd.to_datetime(holdings['portfolio_date'])

nav = nav.sort_values(['amfi_code', 'date']).reset_index(drop=True)
transactions = transactions.sort_values(['investor_id', 'transaction_date']).reset_index(drop=True)

print('Missing values in nav:', nav.isna().sum().sum())
print('Missing values in transactions:', transactions.isna().sum().sum())
print('Missing values in holdings:', holdings.isna().sum().sum())

nav (46000, 3)
transactions (32778, 13)
holdings (322, 8)
fund_master (40, 15)
scheme_performance (40, 19)
Missing values in nav: 0
Missing values in transactions: 0
Missing values in holdings: 0


In [3]:
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()
returns = nav.dropna(subset=['daily_return']).copy()

var_rows = []
for code, group in returns.groupby('amfi_code'):
    ret = group['daily_return'].dropna()
    if len(ret) < 30:
        continue
    var_95 = ret.quantile(VAR_LEVEL)
    cvar_95 = ret[ret <= var_95].mean()
    var_rows.append({
        'amfi_code': int(code),
        'var_95_pct': round(float(-var_95) * 100, 2),
        'cvar_95_pct': round(float(-cvar_95) * 100, 2),
        'daily_return_count': int(len(ret)),
    })

var_cvar_df = pd.DataFrame(var_rows)
var_cvar_df = var_cvar_df.merge(scheme_performance[['amfi_code','scheme_name','fund_house','risk_grade','category']], on='amfi_code', how='left')
var_cvar_df = var_cvar_df.sort_values('var_95_pct', ascending=False).reset_index(drop=True)
var_cvar_df.head()


,amfi_code,var_95_pct,cvar_95_pct,daily_return_count,scheme_name,fund_house,risk_grade,category
0,119599,2.69,3.24,1149,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Very High,Small Cap
1,119095,2.62,3.17,1149,Axis Small Cap Fund - Regular - Growth,Axis Mutual Fund,Very High,Small Cap
2,101207,2.60,3.25,1149,ABSL Small Cap Fund - Regular - Growth,Aditya Birla Sun Life MF,Very High,Small Cap
3,118634,2.54,3.23,1149,Nippon India Small Cap Fund - Regular - Growth,Nippon India MF,Very High,Small Cap
4,119598,2.45,3.06,1149,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Very High,Small Cap


In [4]:
var_cvar_df.to_csv(ROOT / 'var_cvar_report.csv', index=False)
var_cvar_df.head(10)

,amfi_code,var_95_pct,cvar_95_pct,daily_return_count,scheme_name,fund_house,risk_grade,category
0,119599,2.69,3.24,1149,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Very High,Small Cap
1,119095,2.62,3.17,1149,Axis Small Cap Fund - Regular - Growth,Axis Mutual Fund,Very High,Small Cap
2,101207,2.60,3.25,1149,ABSL Small Cap Fund - Regular - Growth,Aditya Birla Sun Life MF,Very High,Small Cap
3,118634,2.54,3.23,1149,Nippon India Small Cap Fund - Regular - Growth,Nippon India MF,Very High,Small Cap
4,119598,2.45,3.06,1149,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Very High,Small Cap
5,149324,2.35,3.10,1149,DSP Small Cap Fund - Regular - Growth,DSP Mutual Fund,Very High,Small Cap
6,102886,1.92,2.33,1149,UTI Mid Cap Fund - Regular - Growth,UTI Mutual Fund,High,Mid Cap
7,100033,1.90,2.35,1149,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,HDFC Mutual Fund,High,Mid Cap
8,120505,1.89,2.43,1149,ICICI Pru Midcap Fund - Regular - Growth,ICICI Prudential MF,High,Mid Cap
9,119094,1.85,2.43,1149,Axis Midcap Fund - Regular - Growth,Axis Mutual Fund,High,Mid Cap


In [5]:
key_funds = [119551, 100016, 120503, 118632, 100033]
fig, ax = plt.subplots(figsize=(12, 6))
for fund_id in key_funds:
    fund_nav = nav[nav['amfi_code'] == fund_id].sort_values('date').copy()
    fund_nav['daily_return'] = fund_nav['nav'].pct_change()
    rolling_mean = fund_nav['daily_return'].rolling(90).mean()
    rolling_std = fund_nav['daily_return'].rolling(90).std(ddof=1)
    rolling_sharpe = rolling_mean / rolling_std * np.sqrt(TRADING_DAYS)
    ax.plot(rolling_sharpe.dropna().index, rolling_sharpe.dropna().values, label=str(fund_id), linewidth=1.5)
ax.set_title('Rolling 90-Day Sharpe Ratio for 5 Key Funds')
ax.set_xlabel('Date')
ax.set_ylabel('Sharpe Ratio')
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(ROOT / 'rolling_sharpe_chart.png', dpi=200)
plt.close(fig)
print('Saved rolling_sharpe_chart.png')

Saved rolling_sharpe_chart.png


In [7]:
transactions['first_transaction_year'] = transactions.groupby('investor_id')['transaction_date'].transform('min').dt.year

sip_transactions = transactions[transactions['transaction_type'] == 'Sip'].copy()
investment_transactions = transactions[transactions['transaction_type'].isin(['Sip', 'Lumpsum'])].copy()

investor_summary = investment_transactions.groupby('investor_id').agg(
    total_invested=('amount_inr', 'sum'),
    investment_count=('amount_inr', 'count'),
).reset_index()

sip_summary = sip_transactions.groupby('investor_id').agg(
    avg_sip_amount=('amount_inr', 'mean'),
    sip_count=('amount_inr', 'count'),
).reset_index()

investor_summary = investor_summary.merge(sip_summary, on='investor_id', how='left')
first_year_map = transactions.drop_duplicates('investor_id')[['investor_id', 'first_transaction_year']].set_index('investor_id')['first_transaction_year']
investor_summary['first_transaction_year'] = investor_summary['investor_id'].map(first_year_map)

fund_preferences = investment_transactions.groupby(['investor_id', 'amfi_code']).size().reset_index(name='transactions_count')
fund_preferences = fund_preferences.sort_values(['investor_id', 'transactions_count'], ascending=[True, False]).drop_duplicates('investor_id')
fund_preferences = fund_preferences[['investor_id', 'amfi_code']].rename(columns={'amfi_code': 'preferred_amfi_code'})
investor_summary = investor_summary.merge(fund_preferences, on='investor_id', how='left')

cohort_summary = investor_summary.groupby('first_transaction_year').agg(
    investors=('investor_id', 'nunique'),
    avg_sip_amount=('avg_sip_amount', 'mean'),
    total_invested=('total_invested', 'sum'),
    sip_count=('sip_count', 'sum'),
).reset_index()

cohort_top_fund = investor_summary.groupby(['first_transaction_year', 'preferred_amfi_code']).size().reset_index(name='investor_count')
cohort_top_fund = cohort_top_fund.sort_values(['first_transaction_year', 'investor_count'], ascending=[True, False]).drop_duplicates('first_transaction_year')
cohort_top_fund = cohort_top_fund.merge(scheme_performance[['amfi_code', 'scheme_name']], left_on='preferred_amfi_code', right_on='amfi_code', how='left')
cohort_summary = cohort_summary.merge(cohort_top_fund[['first_transaction_year', 'scheme_name', 'investor_count']], on='first_transaction_year', how='left')
cohort_summary = cohort_summary.rename(columns={'scheme_name': 'top_fund_preference'})
cohort_summary = cohort_summary.sort_values('total_invested', ascending=False)
cohort_summary.head()

,first_transaction_year,investors,avg_sip_amount,total_invested,sip_count,top_fund_preference,investor_count
0,2024,4770,10952.557684,2258062304,19549.0,HDFC Top 100 Fund - Regular Plan - Growth,422
1,2025,173,13652.023551,18992635,167.0,ABSL Frontline Equity Fund - Regular - Growth,8


In [8]:
sip_only = sip_transactions.sort_values(['investor_id', 'transaction_date'])
continuity_rows = []
for investor, group in sip_only.groupby('investor_id'):
    if len(group) < 6:
        continue
    gaps = group['transaction_date'].diff().dropna().dt.days
    avg_gap = gaps.mean()
    continuity_rows.append({
        'investor_id': investor,
        'n_sips': len(group),
        'avg_gap_days': round(float(avg_gap), 2),
        'at_risk': bool(avg_gap > 35),
    })

continuity_df = pd.DataFrame(continuity_rows)
continuity_df.head()


,investor_id,n_sips,avg_gap_days,at_risk
0,INV000004,6,85.40,True
1,INV000008,6,70.40,True
2,INV000010,6,64.80,True
3,INV000011,7,40.17,True
4,INV000012,8,57.00,True


In [10]:
latest_holdings = holdings.sort_values(['amfi_code', 'portfolio_date']).copy()
latest_holdings['weight_norm'] = latest_holdings.groupby('amfi_code')['weight_pct'].transform(lambda s: s / s.sum())
latest_holdings['hhi_component'] = latest_holdings['weight_norm'] ** 2
sector_hhi_df = latest_holdings.groupby('amfi_code').agg(sector_hhi=('hhi_component', 'sum')).reset_index()
sector_hhi_df = sector_hhi_df.merge(scheme_performance[['amfi_code','scheme_name','fund_house','risk_grade','category']], on='amfi_code', how='left')
sector_hhi_df = sector_hhi_df.sort_values('sector_hhi', ascending=False)
sector_hhi_df.head(10)

,amfi_code,sector_hhi,scheme_name,fund_house,risk_grade,category
11,119092,0.206489,Axis Bluechip Fund - Regular - Growth,Axis Mutual Fund,Moderate,Large Cap
3,101207,0.200741,ABSL Small Cap Fund - Regular - Growth,Aditya Birla Sun Life MF,Very High,Small Cap
18,119599,0.174751,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Very High,Small Cap
4,102885,0.174709,UTI Nifty 50 Index Fund - Regular - Growth,UTI Mutual Fund,Moderate,Index
7,118632,0.168231,Nippon India Large Cap Fund - Regular - Growth,Nippon India MF,Moderate,Large Cap
29,148568,0.167963,Mirae Asset Emerging Bluechip Fund - Regular -...,Mirae Asset MF,Moderately High,Large & Mid Cap
21,120505,0.157507,ICICI Pru Midcap Fund - Regular - Growth,ICICI Prudential MF,High,Mid Cap
22,120506,0.153732,ICICI Pru Value Discovery Fund - Regular - Growth,ICICI Prudential MF,Moderately High,Value
27,125498,0.152414,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,High,Mid Cap
23,120841,0.149650,Kotak Bluechip Fund - Regular - Growth,Kotak Mahindra MF,Moderate,Large Cap


In [11]:
def recommend_funds(risk_appetite):
    risk_map = {'Low': ['Low'], 'Moderate': ['Moderate', 'Moderately High'], 'High': ['High', 'Very High']}
    allowed_risks = risk_map.get(risk_appetite.title(), ['Moderate', 'Moderately High'])
    filtered = scheme_performance[scheme_performance['risk_grade'].isin(allowed_risks)].copy()
    filtered = filtered.sort_values(['sharpe_ratio', 'return_3yr_pct'], ascending=[False, False]).head(3)
    return filtered[['amfi_code', 'scheme_name', 'risk_grade', 'sharpe_ratio']].reset_index(drop=True)

recommend_funds('Moderate')

,amfi_code,scheme_name,risk_grade,sharpe_ratio
0,100016,HDFC Top 100 Fund - Regular Plan - Growth,Moderate,1.06
1,148567,Mirae Asset Large Cap Fund - Regular - Growth,Moderate,1.06
2,120504,ICICI Pru Bluechip Fund - Direct - Growth,Moderate,1.03


## Advanced Insights

1. The highest downside risk among the analyzed funds is observed in the fund with the largest VaR/CVaR estimate, which helps identify the most volatile schemes.
2. Investor cohorts from later years tend to show the highest aggregate capital deployment, revealing how participation evolves over time.
3. SIP continuity analysis highlights the average gap between recurring subscriptions and flags investors whose cadence is becoming unstable.
4. Equity funds with the highest sector HHI indicate portfolios that are more concentrated and less diversified across industries.
5. The recommender ranks the most attractive funds for each risk appetite using the historical Sharpe ratio and risk grade alignment.